# 第4章 主成分分析の計算技法と拡張 ― デモノートブック

第3章では PCA が何を最適化しているかを見た。この章は「ではどう計算するか」に答える。
このノートブックでは、講義ノート §4.1 の表4.1（計算戦略の比較）と例4.1（三つの現場での見積もり）に並ぶ戦略を実際に走らせて
**実測時間と精度**を見る。あわせて欠測・外れ値・解釈性という三つの拡張を数値で確かめる。

**時間計測についての注意**：以下の実行時間は Colab の割り当てられた CPU、BLAS のスレッド数、
同時に走っている他のセッションによって数倍変わる。**絶対値ではなく比と傾きを見ること。**
$O(d^3)$ と $O(n^3)$ の差のような桁の違いはどの環境でも再現するが、
「$55$ 倍速い」といった具体的な倍率は環境依存である。

**データ行列の規約**：$\boldsymbol{X}\in\mathbb{R}^{d\times n}$ は**列がサンプル**。
`scikit-learn` は行がサンプルなので、渡すときに `X.T` と転置する。

## 目次

1. [準備](#setup)
2. [4.1 双対 PCA：$n\ll d$ のときはグラム行列側を解く](#s1)
3. [4.2 ランダム化 SVD：オーバーサンプリングとべき乗反復](#s2)
4. [4.3 オンライン PCA（Oja 則）と増分 PCA](#s3)
5. [4.4 欠測値のある PCA：EM 反復](#s4)
6. [4.5 外れ値 1 点で主軸は何度傾くか](#s5)
7. [4.6 スパース PCA：解釈性と説明力のトレードオフ](#s6)
8. [演習](#ex)
9. [演習の解答](#sol)

## 準備

最初にこのセルを実行する。日本語フォントの設定（Colab には既定で入っていない）と、
以降で使うライブラリの読み込みを行う。フォントの導入に失敗した場合は
図のラベルが自動的に英語に切り替わる（`L()` 関数）。

In [ ]:
import subprocess, sys, warnings
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

warnings.filterwarnings("ignore", category=UserWarning)


def _setup_japanese_font():
    """日本語が出せるフォントを探し、なければ入れる。成功したら True。"""
    cands = ["IPAexGothic", "IPAGothic", "Noto Sans CJK JP", "Noto Sans JP",
             "TakaoGothic", "Yu Gothic", "Hiragino Sans"]
    have = {f.name for f in fm.fontManager.ttflist}
    for name in cands:
        if name in have:
            matplotlib.rcParams["font.family"] = name
            return True
    # Colab 想定：pip で導入する
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "japanize-matplotlib"], check=True, timeout=180)
        import japanize_matplotlib  # noqa: F401  読み込むだけで設定される
        return True
    except Exception:
        pass
    # 予備：apt で IPA フォント
    try:
        subprocess.run("apt-get -qq -y install fonts-ipafont-gothic",
                       shell=True, check=True, timeout=300)
        fm._load_fontmanager(try_read_cache=False)
        matplotlib.rcParams["font.family"] = "IPAGothic"
        return True
    except Exception:
        return False


JP = _setup_japanese_font()


def L(ja, en):
    """日本語フォントが使えれば ja、駄目なら en を返す（図のラベル用）。"""
    return ja if JP else en


matplotlib.rcParams.update({
    "font.size": 11, "axes.titlesize": 12, "axes.labelsize": 11,
    "figure.dpi": 110, "savefig.bbox": "tight",
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.unicode_minus": False,
})

# 講義ノートの図と同じ色
C = {"blue": "#1f4e79", "red": "#c0392b", "green": "#1e8449",
     "orange": "#d68910", "purple": "#6a4c93", "gray": "#7f8c8d"}

print("日本語フォント:", "有効" if JP else "無効（図のラベルは英語になる）")
print("numpy", np.__version__, "| matplotlib", matplotlib.__version__)

<a name="s1"></a>
## 4.1 双対 PCA：$n\ll d$ のときはグラム行列側を解く

定理4.3（双対 PCA）は、グラム行列（式(4.1)）$\boldsymbol{G}=\widetilde{\boldsymbol{X}}^\top\widetilde{\boldsymbol{X}}\in\mathbb{R}^{n\times n}$ の
固有ベクトル $\boldsymbol{v}_i$ から

$$\boldsymbol{u}_i=\frac{1}{\sigma_i}\widetilde{\boldsymbol{X}}\boldsymbol{v}_i\in\mathbb{R}^d,\qquad \sigma_i=\sqrt{\mu_i},\qquad
\widehat{\boldsymbol{\Sigma}}\boldsymbol{u}_i=\frac{\mu_i}{n}\boldsymbol{u}_i$$

と主成分方向が復元できると主張する。$d\times d$ の行列を一度も作らずに済むのが要点である。
まず例4.5（$d=5$, $n=2$ の手計算）を再現する。$\boldsymbol{v}_i$ は**サンプル側** $\mathbb{R}^n$ に、
$\boldsymbol{u}_i$ は**変数側** $\mathbb{R}^d$ に住むことを次元で確かめること。

In [ ]:
# 例4.5：例3.8 と同じ 10 個の数を「5 変数 2 サンプル」として読む
X = np.array([[1., 1.],
              [2., 3.],
              [3., 5.],
              [4., 2.],
              [5., 4.]])                 # d=5（行が変数）, n=2（列がサンプル）
d, n = X.shape
Xt = X - X.mean(axis=1, keepdims=True)
G = Xt.T @ Xt                            # グラム行列（n x n = 2 x 2）

mu, V = np.linalg.eigh(G); mu, V = mu[::-1], V[:, ::-1]
sigma1 = np.sqrt(mu[0])
u1 = Xt @ V[:, 0] / sigma1               # R^n -> R^d へ押し出す（式(4.2)）
if u1[1] > 0:
    u1 = -u1                             # 符号の任意性。本文の向きに合わせる

print("X.shape =", X.shape, " -> d =", d, "変数,", n, "サンプル")
print("Xtilde =\n", Xt)
print("G = Xtilde^T Xtilde =\n", G, "   (", G.shape, ")")
print("G の固有値 mu =", mu, " v1 =", np.round(V[:, 0], 6), " (v1 in R^n, n =", n, ")")
print("u1 = Xtilde v1 / sigma1 =", np.round(u1, 6), " (u1 in R^d, d =", d, ")")
print("sqrt(10) * u1 =", np.round(np.sqrt(10) * u1, 6), " ||u1|| =", round(float(np.linalg.norm(u1)), 12))
print("lambda_1 = mu_1 / n =", mu[0] / n)

# 直接 5x5 の共分散行列を解いても同じ答えになる
S = Xt @ Xt.T / n
lamP, UP = np.linalg.eigh(S); lamP, UP = lamP[::-1], UP[:, ::-1]
print("\n主 PCA（5x5 を対角化）の固有値 =", np.round(lamP, 12))
print("非零固有値の本数 =", int((lamP > 1e-12).sum()), " = min(d, n-1) =", min(d, n - 1))
print("主成分方向の差（符号を揃えて）=",
      np.abs(np.abs(UP[:, 0]) - np.abs(u1)).max())

$\boldsymbol{v}_1\in\mathbb{R}^2$（サンプル側）から $\boldsymbol{u}_1\in\mathbb{R}^5$（変数側）が復元され、
$\sqrt{10}\,\boldsymbol{u}_1=(0,-1,-2,2,1)^\top$、$\lambda_1=\mu_1/n=2.5$ が本文の手計算と一致する。
中心化後の階数は $\min(d,n-1)=1$ なので非零固有値は 1 本だけである
（同じ 10 個の数でも「何を変数と見るか」で解く行列の大きさも主成分の本数も変わる、
という §4.2・注意4.4 の指摘のとおり）。

次に、講義ノートのリスト4.1 の `pca_primal` / `pca_dual` を実装して
**一致の検証**と**実行時間の比較**を行う（図4.1 に対応）。

In [ ]:
import time

def pca_primal(X, k):                      # X is d x n. d x d cov: O(n d^2 + d^3)
    Xt = X - X.mean(axis=1, keepdims=True); n = X.shape[1]
    lam, U = np.linalg.eigh(Xt @ Xt.T / n)             # Sighat = Xt Xt^T / n
    idx = np.argsort(lam)[::-1][:k]
    return lam[idx], U[:, idx]             # principal directions live in R^d

def pca_dual(X, k):                        # n x n Gram: O(n^2 d + n^3)
    Xt = X - X.mean(axis=1, keepdims=True); n = X.shape[1]
    lam, V = np.linalg.eigh(Xt.T @ Xt / n)             # G = Xt^T Xt, eigvecs in R^n
    idx = np.argsort(lam)[::-1][:k]
    lam, V = lam[idx], V[:, idx]
    U = Xt @ V / np.sqrt(n * lam)          # u_i = Xt v_i / sigma_i  -> back to R^d
    return lam, U                          # sigma_i = sqrt(n lambda_i)

def align_sign(A, B):                      # 符号の任意性を参照側 B に合わせる
    return A * np.sign(np.sum(A * B, axis=0))

rng = np.random.default_rng(0)
d0, n0, r = 1200, 60, 4
X = (rng.standard_normal((d0, r)) @ rng.standard_normal((r, n0))
     + 0.5 * rng.standard_normal((d0, n0)))            # d x n、真の階数 4 + ノイズ
l1, U1 = pca_primal(X, 6)
l2, U2 = pca_dual(X, 6)
print(f"d={d0}, n={n0}、真の階数 {r} での一致の検証")
print("  固有値 lambda_1..6     =", np.round(l1, 4))
print("  固有値の最大差         =", f"{np.abs(l1 - l2).max():.2e}")
print("  主成分方向の差（成分別）=",
      np.array([f"{v:.1e}" for v in np.abs(align_sign(U2, U1) - U1).max(axis=0)]))
print("  上位4本の射影行列の差  =",
      f"{np.abs(U1[:, :4] @ U1[:, :4].T - U2[:, :4] @ U2[:, :4].T).max():.2e}")
print("  上位6本の射影行列の差  =",
      f"{np.abs(U1 @ U1.T - U2 @ U2.T).max():.2e}")

# 実行時間の比較（環境によって数倍変わる。比と傾きを見ること）
ds, t_pri, t_dua = [250, 500, 1000, 2000], [], []
for dd in ds:
    Xd = (rng.standard_normal((dd, r)) @ rng.standard_normal((r, 50))
          + 0.5 * rng.standard_normal((dd, 50)))
    t0 = time.perf_counter(); pca_primal(Xd, 3); t_pri.append(time.perf_counter() - t0)
    t0 = time.perf_counter(); pca_dual(Xd, 3);   t_dua.append(time.perf_counter() - t0)
    print(f"  d={dd:>5}, n=50: 主 PCA {t_pri[-1]*1e3:8.2f} ms,"
          f"  双対 PCA {t_dua[-1]*1e3:7.2f} ms,  比 {t_pri[-1]/t_dua[-1]:7.1f} 倍")

fig, ax = plt.subplots(figsize=(6.4, 4.2))
ax.loglog(ds, np.array(t_pri) * 1e3, "o-", color=C["red"], lw=2,
          label=L(r"主 PCA（$d\times d$ を分解）", r"primal PCA ($d\times d$)"))
ax.loglog(ds, np.array(t_dua) * 1e3, "s-", color=C["blue"], lw=2,
          label=L(r"双対 PCA（$n\times n$ を分解）", r"dual PCA ($n\times n$)"))
ref = np.array(ds, float) ** 3; ref = ref / ref[0] * t_pri[0] * 1e3
ax.loglog(ds, ref, ":", color=C["gray"], lw=1.6,
          label=L(r"$O(d^3)$ の傾き（参考）", r"$O(d^3)$ slope (ref.)"))
ax.set_xlabel(L("変数の数 $d$（$n=50$ 固定）", "number of variables $d$ ($n=50$)"))
ax.set_ylabel(L("実行時間 (ms)", "runtime (ms)"))
ax.set_title(L("主 PCA と双対 PCA の計算時間", "runtime: primal vs dual PCA"))
ax.legend(fontsize=9); fig.tight_layout(); plt.show()

固有値と（符号を揃えた）固有ベクトルは倍精度の丸め誤差の範囲で一致する。
ただし成分別に見ると**第 5・第 6 主成分だけ 2 桁ほど一致が悪い**（$10^{-16}$ 台に対し $10^{-14}$ 台）。
真の階数が 4 なので $\lambda_5\approx7.12$、$\lambda_6\approx7.03$ はノイズ由来でほぼ縮退しており、
ギャップ $\delta=\lambda_5-\lambda_6$ が小さいぶん、定理4.25（Davis--Kahan）の右辺
$2\|\boldsymbol{E}\|_F/\delta$ が大きくなって個々の方向が定まらなくなるからである。
いっぽう**射影行列**（＝部分空間）は上位 4 本でも上位 6 本でも一致している。
個々のベクトルではなく部分空間で比較するのが正しい作法である（§4.9 の落とし穴 2、例4.24、演習 2）。

時間は $d$ を 8 倍（$250\to2000$）にすると主 PCA が 100 倍以上に伸びる一方、
双対 PCA は $n=50$ 固定なので $O(n^2d)$ でほとんど増えず、1 ミリ秒前後にとどまる。
$d=2000$ で 300 倍以上の差である。実測の傾きは参考に引いた $O(d^3)$ の直線より緩いが、
これはこの範囲では BLAS の並列化とメモリ帯域が効いているためで、$d$ を上げるほど 3 次に近づく。
$n\ll d$ の領域（バイオインフォマティクス、画像、テキストの多く）では双対 PCA が唯一現実的な選択肢になる。

<a name="s2"></a>
## 4.2 ランダム化 SVD：オーバーサンプリングとべき乗反復

§4.4 のアルゴリズム（図4.2 とリスト4.2）を実装する。パラメータは目標階数 $k$、
オーバーサンプリング $p$、べき乗反復 $q$ の三つである。定理4.9（Halko--Martinsson--Tropp）は
$q=0$ で

$$\mathbb{E}\|\boldsymbol{X}-\boldsymbol{Q}\boldsymbol{Q}^\top\boldsymbol{X}\|_F\le\Bigl(1+\frac{k}{p-1}\Bigr)^{1/2}
\Bigl(\sum_{i>k}\sigma_i^2\Bigr)^{1/2}$$

を保証する（式(4.4)。作用素ノルム版は式(4.5)）。右辺の後半は Eckart--Young の下限
（階数 $k$ のどんな近似もこれより良くならない）なので、
係数 $(1+k/(p-1))^{1/2}$ が**最適からの離れ具合**である。
また注意4.10 のとおり、べき乗反復は特異値を $\sigma_i^{2q+1}$ に置き換えて減衰を鋭くする。

特異値の減衰が**速い**行列（$\sigma_i=0.8^{i-1}$）と**遅い**行列（$\sigma_i=1/i$）の両方で、
$q$ の効き方の違いを見る。

In [ ]:
def randomized_svd(X, k, p=10, q=2, seed=0):         # X is d x n
    rng = np.random.default_rng(seed)
    Om = rng.standard_normal((X.shape[1], k + p))    # step 1: Omega is n x (k+p)
    Q, _ = np.linalg.qr(X @ Om)                      # step 2: Y = X Om is d x (k+p)
    for _ in range(q):                               # step 3: 毎回 QR で再直交化する
        Q, _ = np.linalg.qr(X.T @ Q)
        Q, _ = np.linalg.qr(X @ Q)
    B = Q.T @ X                                      # step 4: (k+p) x n
    Ub, s, Vt = np.linalg.svd(B, full_matrices=False)
    return (Q @ Ub)[:, :k], s[:k], Vt[:k], Q         # step 5: U の列は R^d の元

def make_matrix(kind, d=1200, n=800, m=200, seed=0):
    rng = np.random.default_rng(seed)
    Ud, _ = np.linalg.qr(rng.standard_normal((d, m)))     # 変数側 R^d
    Vn, _ = np.linalg.qr(rng.standard_normal((n, m)))     # サンプル側 R^n
    sv = 0.8 ** np.arange(m) if kind == "fast" else 1.0 / np.arange(1, m + 1)
    return (Ud * sv) @ Vn.T                               # d x n

k = 10
err, times = {}, {}
for kind in ("fast", "slow"):
    A = make_matrix(kind)
    t0 = time.perf_counter(); s_exact = np.linalg.svd(A, compute_uv=False)
    t_exact = time.perf_counter() - t0
    lab = L("速い減衰 $0.8^{i-1}$", "fast decay") if kind == "fast" else L("遅い減衰 $1/i$", "slow decay")
    print(f"[{kind}] 厳密 SVD: {t_exact*1e3:.1f} ms,  sigma_1..3 =", np.round(s_exact[:3], 5))
    err[kind], times[kind] = [], []
    for q in (0, 1, 2, 3):
        t0 = time.perf_counter(); _, s_r, _, Q = randomized_svd(A, k, p=10, q=q)
        dt = time.perf_counter() - t0
        rel = np.max(np.abs(s_r - s_exact[:k]) / s_exact[:k])
        err[kind].append(rel); times[kind].append(dt)
        print(f"    q={q}: 上位{k}本の最大相対誤差 = {rel:.2e},  {dt*1e3:6.1f} ms"
              f"  （厳密 SVD の {t_exact/dt:5.1f} 倍速い）")

# オーバーサンプリング p の効果（q=0）と理論係数の比較
A = make_matrix("fast")
s_exact = np.linalg.svd(A, compute_uv=False)
tail = np.sqrt((s_exact[k:] ** 2).sum())
print("\nEckart--Young の下限 (sum_{i>k} sigma_i^2)^{1/2} =", f"{tail:.4e}")
ps, meas, theo, rk = [2, 5, 10, 20, 40], [], [], []
for p in ps:
    Uk, sk, Vk, Q = randomized_svd(A, k, p=p, q=0)
    res = np.linalg.norm(A - Q @ (Q.T @ A), "fro")          # Q の列は k+p 本
    rec = np.linalg.norm(A - (Uk * sk) @ Vk, "fro")         # 階数 k に打ち切った再構成
    meas.append(res / tail); theo.append(np.sqrt(1 + k / (p - 1))); rk.append(rec / tail)
    print(f"  p={p:>2}: ||A - QQ^T A||_F / 下限 = {meas[-1]:.4f}（理論界"
          f" {theo[-1]:.4f} 以下）,  階数 k の再構成 / 下限 = {rk[-1]:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
ax = axes[0]
for kind, col, mk in [("fast", C["blue"], "o"), ("slow", C["red"], "s")]:
    ax.semilogy([0, 1, 2, 3], err[kind], mk + "-", color=col, lw=2,
                label=L("速い減衰 $\\sigma_i=0.8^{i-1}$", "fast decay $0.8^{i-1}$") if kind == "fast"
                else L("遅い減衰 $\\sigma_i=1/i$", "slow decay $1/i$"))
ax.set_xticks([0, 1, 2, 3])
ax.set_xlabel(L("べき乗反復の回数 $q$", "power iterations $q$"))
ax.set_ylabel(L("上位10本の最大相対誤差", "max rel. error of top 10"))
ax.set_title(L("べき乗反復の効果は減衰の速さで変わる", "effect of power iteration vs decay"))
ax.legend(fontsize=9)

ax = axes[1]
ax.plot(ps, meas, "o-", color=C["blue"], lw=2,
        label=L(r"実測 $\|A-QQ^\top A\|_F$", r"measured $\|A-QQ^\top A\|_F$"))
ax.plot(ps, theo, "s--", color=C["gray"], lw=2, label=L("定理4.9 の界", "bound of Thm 4.9"))
ax.plot(ps, rk, "^-", color=C["orange"], lw=2,
        label=L("階数 $k$ に打ち切った再構成", "rank-$k$ reconstruction"))
ax.axhline(1.0, color=C["green"], ls=":", lw=1.6,
           label=L("Eckart--Young の最適値", "Eckart-Young optimum"))
ax.set_yscale("log")
ax.set_xlabel(L("オーバーサンプリング $p$", "oversampling $p$"))
ax.set_ylabel(L("最適値に対する比", "ratio to the optimum"))
ax.set_title(L("$q=0$ でのオーバーサンプリングの効果", "effect of oversampling ($q=0$)"))
ax.legend(fontsize=8)
fig.tight_layout(); plt.show()

減衰の速い行列では $q$ を 1 増やすごとに誤差が数桁改善する。
これは注意4.10 のとおり残差の比が $(\sigma_{k+1}/\sigma_k)^{2q+1}$ で縮むからで、
$\sigma_{k+1}/\sigma_k=0.8$ なら $q=2$ で指数 5、$0.8^5=0.328$ の**さらに累乗**の効果が出る。
いっぽう $\sigma_i=1/i$ の行列では $\sigma_{11}/\sigma_{10}=10/11=0.909$ で減衰が遅い。
$q=0$ の誤差が $20\%$ もあるので**べき乗反復は必須**だが、1 回あたりの改善は $0.909^{2q+1}$ と鈍く、
$q=3$ まで回しても $4.6\times10^{-6}$——速い減衰の $q=1$（$7.3\times10^{-8}$）にすら届かない。
この場合はさらに $q$ を増やすより $k+p$ を増やすほうが効く（演習4.6(1)）。

$p$ の効果は右の図である。実測の $\|\boldsymbol{X}-\boldsymbol{Q}\boldsymbol{Q}^\top\boldsymbol{X}\|_F$ はどの $p$ でも
定理4.9 の界の内側にあり、$p$ を増やすと界も実測も小さくなる。
ここで注意したいのは、$p\ge5$ で実測が Eckart--Young の最適値（比 1）を**下回る**ことである。
矛盾ではない。$\boldsymbol{Q}$ の列は $k+p$ 本あるので $\boldsymbol{Q}\boldsymbol{Q}^\top\boldsymbol{X}$ の階数は $k$ より大きく、
階数 $k$ の最良近似と比べる相手ではないからである。実際に階数 $k$ へ打ち切った再構成（橙）は
つねに 1 以上で、$p$ を増やすと 1 に近づく。
なお定理4.9 の界は**期待値**に対する評価なので、個々の実現が界の中に入ることまでは
保証していない（講義ノート 演習4.6(4) の注意）。

実行時間は厳密な SVD より 1〜2 桁速いが、**具体的な倍率は環境依存**である。
実務では `sklearn.utils.extmath.randomized_svd` や `PCA(svd_solver='randomized')` が
同じアルゴリズムを提供する。

<a name="s3"></a>
## 4.3 オンライン PCA（Oja 則）と増分 PCA

定義4.11・式(4.6) の Oja の学習則

$$\boldsymbol{w}_t\leftarrow\boldsymbol{w}_{t-1}+\eta_t\bigl(\boldsymbol{x}_t\boldsymbol{x}_t^\top\boldsymbol{w}_{t-1}
-(\boldsymbol{w}_{t-1}^\top\boldsymbol{x}_t\boldsymbol{x}_t^\top\boldsymbol{w}_{t-1})\boldsymbol{w}_{t-1}\bigr)$$

は 1 サンプルあたり $O(d)$ の計算とメモリしか使わず、データ行列を一度も保持しない（§4.5.1）。
定理4.13 によれば平均場の ODE（式(4.8)） $\dot{\boldsymbol{w}}=\boldsymbol{\Sigma}\boldsymbol{w}-(\boldsymbol{w}^\top\boldsymbol{\Sigma}\boldsymbol{w})\boldsymbol{w}$ の
漸近安定な平衡点は最大固有値の固有ベクトルだけである。次を確かめる。

1. 学習率 $\eta_t=\eta_0/(t_0+t)$（Robbins--Monro 条件）と定数 $\eta$ の違い。
2. 命題4.12 の**復元力**——正規化を外しても $\|\boldsymbol{w}\|$ が 1 付近に戻ること。
3. 収束の速さが $\lambda_1-\lambda_2$ に支配されること（定理4.13）。

In [ ]:
def oja(Xs, eta, w0, q1, normalize=True):
    """Xs は d x T（列が到着するサンプル）。overlap とノルムの履歴、最終の w を返す。"""
    w = w0.copy()
    hist = np.empty(Xs.shape[1]); nrm = np.empty(Xs.shape[1])
    for t in range(Xs.shape[1]):
        x = Xs[:, t]
        g = float(x @ w)
        e = eta(t) if callable(eta) else eta
        w = w + e * g * (x - g * w)                 # (I - w w^T) x x^T w の形
        if normalize:
            w /= np.linalg.norm(w)
        nrm[t] = np.linalg.norm(w); hist[t] = abs(w @ q1) / nrm[t]
    return hist, nrm, w

rng = np.random.default_rng(0)
d, T = 10, 20000
Qrot, _ = np.linalg.qr(rng.standard_normal((d, d)))
lamT = np.r_[5.0, 2.0, np.full(d - 2, 1.0)]         # 固有値を明示的に置く
Sigma = (Qrot * lamT) @ Qrot.T
q1 = Qrot[:, 0]                                     # 真の第1固有ベクトル
Lchol = np.linalg.cholesky(Sigma + 1e-12 * np.eye(d))
Xs = Lchol @ rng.standard_normal((d, T))            # d x T、平均 0、共分散 Sigma

print("真の固有値 =", lamT, " lambda_1/lambda_2 =", lamT[0] / lamT[1])
w0 = rng.standard_normal(d); w0 /= np.linalg.norm(w0)
h_rm, _, w_rm = oja(Xs, lambda t: 1.0 / (100 + t), w0, q1)
h_const, _, w_const = oja(Xs, 0.01, w0, q1)
print("Robbins--Monro 型 eta_t = 1/(100+t): 最終 overlap |<w,q1>| =", round(float(h_rm[-1]), 6))
print("定数 eta = 0.01                    : 最終 overlap |<w,q1>| =", round(float(h_const[-1]), 6))
print("  末尾1000ステップの overlap: Robbins--Monro 平均",
      f"{h_rm[-1000:].mean():.5f} 標準偏差 {h_rm[-1000:].std():.2e}",
      " / 定数 平均", f"{h_const[-1000:].mean():.5f} 標準偏差 {h_const[-1000:].std():.2e}")

# 命題4.12：正規化を外しても ||w|| は 1 付近に戻る（復元力）
print("\n正規化なし（定数 eta=0.01）の ||w|| の推移")
for r0 in (0.5, 1.0, 2.0, 5.0):
    _, nrm, _ = oja(Xs[:, :5000], 0.01, w0 * r0, q1, normalize=False)
    print(f"  初期ノルム {r0:>4.1f} -> 100 ステップ後 {nrm[99]:.3f} -> 最終 {nrm[-1]:.3f}")

# 収束の速さは lambda_1 - lambda_2 に支配される（定理4.13）
print("\n固有値ギャップと収束（eta_t = 1/(100+t)、5000 サンプル）")
gap_hist = {}
e1 = np.eye(d)[:, 0]
for ratio in (1.1, 2.0, 10.0):
    sd = np.sqrt(np.r_[ratio, 1.0, np.full(d - 2, 0.5)])
    Xg = sd[:, None] * rng.standard_normal((d, 5000))        # 対角共分散、真の方向は e_1
    h, _, _ = oja(Xg, lambda t: 1.0 / (100 + t), w0, e1)
    gap_hist[ratio] = h
    print(f"  lambda_1/lambda_2 = {ratio:>4.1f}: 最終 overlap = {h[-1]:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
ax = axes[0]
ax.plot(1 - h_rm, color=C["blue"], lw=1.2, label=L(r"$\eta_t=1/(100+t)$", r"$\eta_t=1/(100+t)$"))
ax.plot(1 - h_const, color=C["red"], lw=1.2, alpha=0.8,
        label=L(r"$\eta=0.01$（定数）", r"$\eta=0.01$ (constant)"))
ax.set_yscale("log"); ax.set_xlabel(L("到着したサンプル数 $t$", "samples seen $t$"))
ax.set_ylabel(L(r"$1-|\langle w_t,q_1\rangle|$", r"$1-|\langle w_t,q_1\rangle|$"))
ax.set_title(L("学習率の設計", "choice of learning rate")); ax.legend(fontsize=9)

ax = axes[1]
for ratio, col in zip((1.1, 2.0, 10.0), (C["gray"], C["orange"], C["green"])):
    ax.plot(1 - gap_hist[ratio], color=col, lw=1.3,
            label=L(rf"$\lambda_1/\lambda_2={ratio}$", rf"$\lambda_1/\lambda_2={ratio}$"))
ax.set_yscale("log"); ax.set_xlabel(L("到着したサンプル数 $t$", "samples seen $t$"))
ax.set_ylabel(L(r"$1-|\langle w_t,q_1\rangle|$", r"$1-|\langle w_t,q_1\rangle|$"))
ax.set_title(L("ギャップが収束を支配する", "the gap governs convergence")); ax.legend(fontsize=9)
fig.tight_layout(); plt.show()

Robbins--Monro 型の学習率では真の第 1 固有ベクトルとの内積が末尾 1000 ステップで
平均 $0.99963$、標準偏差 $7.3\times10^{-5}$ に落ち着く。
定数学習率だと最適解の周りで揺らぎ続け（平均 $0.96487$、標準偏差 $1.8\times10^{-2}$）、
$O(\eta_0)$ の定常揺らぎが残って収束しない。

正規化を外しても $\|\boldsymbol{w}\|$ は初期値 $0.5,1,2,5$ のどこから始めても 1 付近に戻る。
一般の $r=\|\boldsymbol{w}\|$ に対する式(4.7) の一次項 $2\eta g^2(1-r^2)$ が $r>1$ で負、$r<1$ で正になるからで、
明示的な正規化は数値的な安全策であって収束のために不可欠なわけではない。

右の図は固有値比を変えた実験である。$\lambda_1/\lambda_2=10$ なら数千サンプルで収束するが、
$1.1$ では 5000 サンプルでもまだ収束しきらない。定理4.13 の
$\mathrm{e}^{(\lambda_2-\lambda_1)t}$ という予言のとおり、収束の指数はギャップに比例する。

Oja 則は 1 本ずつしか扱えず学習率の設計も要る。実務でより使われるのは §4.5.2 の**増分 PCA**で、
ミニバッチごとに「これまでの要約」と「新しいバッチ」を SVD で融合する。
`scikit-learn` の `IncrementalPCA` で全データを一度に見る `PCA` と比べる。

In [ ]:
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA, IncrementalPCA

digits = load_digits()
Xd = digits.data.T                        # d=64 x n=1797（列がサンプル）
k = 10
full = PCA(n_components=k).fit(Xd.T)      # sklearn は n x d 規約なので転置
inc = IncrementalPCA(n_components=k)
for i in range(0, Xd.shape[1], 200):      # 200 点ずつ partial_fit（ミニバッチ）
    inc.partial_fit(Xd[:, i:i + 200].T)

print("d, n =", Xd.shape)
print("寄与率（全データ一括）=", np.round(full.explained_variance_ratio_[:5], 5))
print("寄与率（増分 PCA）    =", np.round(inc.explained_variance_ratio_[:5], 5))
print("寄与率の最大差        =",
      f"{np.abs(full.explained_variance_ratio_ - inc.explained_variance_ratio_).max():.2e}")

# 部分空間としての一致を主角（principal angles）で測る
Uf, Ui = full.components_.T, inc.components_.T           # d x k（列が主成分方向）
sv = np.linalg.svd(Uf.T @ Ui, compute_uv=False)
ang = np.degrees(np.arccos(np.clip(sv, -1, 1)))
print("上位10次元部分空間の主角（度）=", np.round(ang, 4))
print("個々の主成分方向の |内積| =", np.round(np.abs(np.sum(Uf * Ui, axis=0)), 5))

増分 PCA は 200 点ずつしか見ていないのに、寄与率は全データ一括の `PCA` と
最大差 $8.2\times10^{-4}$ で一致する。上位 5 次元ぶんの主角は $1^\circ$ 未満だが、
6 番目以降は $3^\circ$〜$9^\circ$ とずれが大きくなる。固有値が小さく互いに近い成分では
方向が定まりにくいからで、これも §4.9 の落とし穴 2（例4.24：近い固有値の固有ベクトルは不安定）である。
要約を階数 $k$ に打ち切る誤差は $\sigma_{k+1}$ 以下を捨てているだけなので、
スペクトルが減衰していれば上位成分への実害は小さい。

<a name="s4"></a>
## 4.4 欠測値のある PCA：EM 反復

まず例4.15 の**平均補完のバイアス**を再現する。欠測率 $q$ で平均補完すると分散と共分散が
およそ $(1-q)$ 倍に縮むので、相関は $\rho\sqrt{1-q}$ になるはずである。

次に §4.6 の交互反復（M ステップ＝中心化して階数 $k$ の最良近似、E ステップ＝欠測成分だけを
$(\bar{\boldsymbol{x}}\boldsymbol{1}^\top+\widehat{\boldsymbol{X}}_k)_{ij}$ に置き換え）を実装し、
命題4.16 の主張——目的関数（式(4.9)）

$$J(\widehat{\boldsymbol{X}},\boldsymbol{m},\boldsymbol{L})=\|\widehat{\boldsymbol{X}}-\boldsymbol{m}\boldsymbol{1}^\top-\boldsymbol{L}\|_F^2,
\qquad \operatorname{rank}\boldsymbol{L}\le k,\ \boldsymbol{L}\boldsymbol{1}=\boldsymbol{0}$$

が各ステップで非増加であること——を数値で確かめる。

In [ ]:
# (1) 平均補完のバイアス（例4.15）
rho, n = 0.9, 2000
rng = np.random.default_rng(0)
Lm = np.array([[1.0, 0.0], [rho, np.sqrt(1 - rho ** 2)]])
X = Lm @ rng.standard_normal((2, n))                      # d=2 x n
print("完全データの標本相関 =", round(float(np.corrcoef(X)[0, 1]), 4))
print(" q    平均補完後の相関   理論値 rho*sqrt(1-q)   第1主成分の寄与率")
for q in (0.1, 0.3, 0.5, 0.7):
    Xi = X.copy()
    miss = rng.random(n) < q                              # 第2行（第2変数）を MCAR 欠測
    Xi[1, miss] = Xi[1, ~miss].mean()                     # 観測値の平均で埋める
    lam = np.sort(np.linalg.eigvalsh(np.cov(Xi, bias=True)))[::-1]
    print(f"{q:>4.1f}      {np.corrcoef(Xi)[0,1]:>7.4f}            {rho*np.sqrt(1-q):>7.4f}"
          f"            {lam[0]/lam.sum():>7.4f}")

# (2) EM 反復（§4.6、命題4.16）
def em_lowrank(Xobs, mask, k, n_iter=20):
    """mask=True が観測。J の履歴と補完後の行列を返す。"""
    Xh = Xobs.copy()
    row_mean = np.array([Xobs[i, mask[i]].mean() for i in range(Xobs.shape[0])])
    Xh[~mask] = np.repeat(row_mean[:, None], Xobs.shape[1], axis=1)[~mask]   # 行平均で初期化
    Js = []
    for _ in range(n_iter):
        m = Xh.mean(axis=1)                                # M ステップ：最適なオフセット
        Xt = Xh - m[:, None]
        Uu, ss, Vt = np.linalg.svd(Xt, full_matrices=False)
        Lk = (Uu[:, :k] * ss[:k]) @ Vt[:k]                 # 階数 k の最良近似（Eckart--Young）
        Js.append(float(np.linalg.norm(Xh - m[:, None] - Lk, "fro") ** 2))
        Xh = np.where(mask, Xobs, m[:, None] + Lk)         # E ステップ：欠測成分だけ置換
    return np.array(Js), Xh, m, Lk

rng = np.random.default_rng(1)
dE, nE, kE = 6, 200, 2
Wt = rng.standard_normal((dE, kE))
Xtrue = Wt @ rng.standard_normal((kE, nE)) + 0.3 * rng.standard_normal((dE, nE))
mask = rng.random((dE, nE)) > 0.25                          # 欠測率 25%
Xobs = np.where(mask, Xtrue, np.nan)
print("\n欠測率 =", round(float(1 - mask.mean()), 4))

Js, Xh, m, Lk = em_lowrank(np.nan_to_num(Xobs), mask, kE, 20)
print("J の推移 =", np.round(Js[:6], 2), "...", np.round(Js[-2:], 2))
print("J は単調非増加か:", bool(np.all(np.diff(Js) <= 1e-9)),
      " 減少量の合計 =", round(float(Js[0] - Js[-1]), 3))
print("代入行列 xbar 1^T + Xhat_k の階数 =",
      int(np.linalg.matrix_rank(m[:, None] + Lk, tol=1e-8)), "（k =", kE, "なので k+1）")

# 部分空間の回復精度を平均補完と比べる
def subspace_err(Xa, k, Utrue):
    Xt = Xa - Xa.mean(axis=1, keepdims=True)
    U = np.linalg.svd(Xt, full_matrices=False)[0][:, :k]
    return np.linalg.norm(U @ U.T - Utrue @ Utrue.T, "fro")

Utrue = np.linalg.svd(Xtrue - Xtrue.mean(axis=1, keepdims=True),
                      full_matrices=False)[0][:, :kE]
Xmean = np.nan_to_num(Xobs).copy()
rm = np.array([Xtrue[i, mask[i]].mean() for i in range(dE)])
Xmean[~mask] = np.repeat(rm[:, None], nE, axis=1)[~mask]
print("部分空間の誤差 ||UU^T - U*U*^T||_F : 平均補完 =", round(subspace_err(Xmean, kE, Utrue), 4),
      " EM =", round(subspace_err(Xh, kE, Utrue), 4))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
ax = axes[0]
qs = np.linspace(0.02, 0.8, 25)
emp = []
for q in qs:
    Xi = X.copy()
    miss = np.random.default_rng(7).random(X.shape[1]) < q
    Xi[1, miss] = Xi[1, ~miss].mean()
    emp.append(np.corrcoef(Xi)[0, 1])
ax.plot(qs, emp, "o-", ms=3.5, color=C["red"], lw=1.6, label=L("平均補完の実測", "mean imputation"))
ax.plot(qs, rho * np.sqrt(1 - qs), "--", color=C["gray"], lw=2,
        label=L(r"理論値 $\rho\sqrt{1-q}$", r"theory $\rho\sqrt{1-q}$"))
ax.axhline(rho, color=C["blue"], ls=":", lw=1.6, label=L(r"真の相関 $\rho=0.9$", r"true $\rho=0.9$"))
ax.set_xlabel(L("欠測率 $q$", "missing rate $q$")); ax.set_ylabel(L("標本相関", "sample correlation"))
ax.set_title(L("平均補完は相関構造を歪める", "mean imputation distorts correlation"))
ax.legend(fontsize=9)

ax = axes[1]
ax.plot(np.arange(1, len(Js) + 1), Js, "o-", color=C["blue"], lw=2)
ax.set_yscale("log"); ax.set_xlabel(L("反復回数", "iteration"))
ax.set_ylabel(L("目的関数 $J$", "objective $J$"))
ax.set_title(L("EM 反復で $J$ は単調に減る（命題4.16）", "$J$ decreases monotonically"))
fig.tight_layout(); plt.show()

平均補完後の相関は理論値 $\rho\sqrt{1-q}$ とよく一致し、欠測率が上がるほど相関が失われる。
$q=0.5$ なら $0.9\times0.707=0.636$ 前後で、講義ノート 例4.15 の実測 $0.639$ と同じ現象である。
補完した点は $x_2=\bar x_2$ の直線上に集まるので第 2 変数の分散が過小評価され、
共分散も（埋めた分の寄与が 0 なので）過小評価される。

EM 反復では $J$ が単調に減少する（右図、対数軸）。命題4.16 の交互最小化としての解釈のとおりである。
代入行列 $\bar{\boldsymbol{x}}\boldsymbol{1}^\top+\widehat{\boldsymbol{X}}_k$ の階数は $k+1=3$ で、
「階数 $k$ 以下の行列への近似」という目的関数ではこの反復を記述できない
（平均項と低階数項を分けて書く必要がある）という §4.6・注意4.2 の指摘も数値で確認できる。
部分空間の回復も平均補完より EM のほうが良い。

ただし保証されるのは $J$ の単調減少と収束だけで、到達点が大域最適とは限らない（初期値依存）。
欠測率が高いときはとくに注意がいる。

<a name="s5"></a>
## 4.5 外れ値 1 点で主軸は何度傾くか

例4.17 を再現する。標本共分散の**破綻点は 0** であり、汚染の割合をどれだけ小さくしても、
外れ値を十分遠くに置けば推定量を任意に狂わせられる。
外れ値 $\boldsymbol{x}_0$ の共分散への寄与は $\frac1n\boldsymbol{x}_0\boldsymbol{x}_0^\top$、すなわち $\|\boldsymbol{x}_0\|^2$ に比例するので、
$\|\boldsymbol{x}_0\|=30$、他の点が $\|\boldsymbol{x}\|\approx2$ なら 1 点で $(30/2)^2=225$ 点分の重みを持つ。

あわせて演習4.8(4) の簡易ロバスト推定（マハラノビス距離が下位 80% の点だけを使って 2 回反復）を
実装して比べる（図4.3 に対応）。

In [ ]:
def pc1_angle(X):
    """第1主成分方向が x1 軸となす角（度、0〜90）"""
    Xt = X - X.mean(axis=1, keepdims=True)
    U = np.linalg.svd(Xt, full_matrices=False)[0]
    a = np.degrees(np.arctan2(abs(U[1, 0]), abs(U[0, 0])))
    return a

def robust_pc1_angle(X, keep=0.8, n_iter=2):
    """マハラノビス距離が下位 keep の点だけで平均・共分散を再推定（2 回反復）"""
    idx = np.arange(X.shape[1])
    for _ in range(n_iter):
        Xs = X[:, idx]
        mu = Xs.mean(axis=1, keepdims=True)
        S = np.cov(Xs, bias=True) + 1e-12 * np.eye(X.shape[0])
        dd = np.einsum("ij,ij->j", X - mu, np.linalg.solve(S, X - mu))
        idx = np.argsort(dd)[:int(keep * X.shape[1])]
    return pc1_angle(X[:, idx]), idx

rng = np.random.default_rng(0)
n = 200
X0 = np.diag([2.0, 0.5]) @ rng.standard_normal((2, n))     # 共分散 diag(4, 0.25)
print("外れ値なし: 第1主成分の角度 =", f"{pc1_angle(X0):.2f}", "度（真の主軸は x1 軸）")
X1 = np.column_stack([X0, [0.0, 30.0]])                    # 1 点 (0,30) を加える
print("外れ値 1 点: 第1主成分の角度 =", f"{pc1_angle(X1):.2f}",
      "度  → ほぼ x2 軸方向（0.5% の汚染で 85 度回る）")

ms = [0, 1, 2, 5, 10]
ang_plain, ang_rob = [], []
for m in ms:
    Xm = X0.copy()
    if m:
        jit = np.linspace(-0.1, 0.1, m) if m > 1 else np.zeros(1)
        th = np.pi / 2 + jit                               # 半径 30、ほぼ x2 方向に m 点
        Xm = np.column_stack([X0, 30.0 * np.vstack([np.cos(th), np.sin(th)])])
    ang_plain.append(pc1_angle(Xm))
    ang_rob.append(robust_pc1_angle(Xm)[0])
    print(f"  外れ値 {m:>2} 点（全体の {m/(n+m)*100:4.1f}%）: 通常の PCA {ang_plain[-1]:6.2f} 度,"
          f"  簡易ロバスト {ang_rob[-1]:5.2f} 度")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
ax = axes[0]
ax.scatter(X0[0], X0[1], s=14, color=C["gray"], alpha=0.8, label=L("多数派", "bulk"))
ax.scatter([0], [30], s=90, marker="*", color=C["red"], zorder=5, label=L("外れ値", "outlier"))
for X_, col, lab in [(X0, C["blue"], L("外れ値なしの主軸", "PC1 without outlier")),
                     (X1, C["red"], L("外れ値ありの主軸", "PC1 with outlier"))]:
    Xt = X_ - X_.mean(axis=1, keepdims=True)
    u = np.linalg.svd(Xt, full_matrices=False)[0][:, 0]
    ax.plot([-12 * u[0], 12 * u[0]], [-12 * u[1], 12 * u[1]], color=col, lw=2.2, label=lab)
idxr = robust_pc1_angle(X1)[1]
Xr = X1[:, idxr]; Xtr = Xr - Xr.mean(axis=1, keepdims=True)
ur = np.linalg.svd(Xtr, full_matrices=False)[0][:, 0]
ax.plot([-12 * ur[0], 12 * ur[0]], [-12 * ur[1], 12 * ur[1]], color=C["green"], ls="--", lw=2.2,
        label=L("簡易ロバスト推定", "trimmed estimate"))
ax.set_xlim(-9, 9); ax.set_ylim(-6, 32); ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$")
ax.set_title(L("1 点の外れ値で主軸が倒れる", "one outlier tilts the principal axis"))
ax.legend(fontsize=8, loc="upper left")

ax = axes[1]
ax.plot(ms, ang_plain, "o-", color=C["red"], lw=2, label=L("通常の PCA", "plain PCA"))
ax.plot(ms, ang_rob, "s--", color=C["green"], lw=2, label=L("簡易ロバスト推定", "trimmed estimate"))
ax.set_xlabel(L("外れ値の点数 $m$", "number of outliers $m$"))
ax.set_ylabel(L("真の主軸とのなす角（度）", "angle from true axis (deg)"))
ax.set_title(L("破綻点は 0（1 点で壊れる）", "breakdown point is zero"))
ax.set_ylim(-3, 93); ax.legend(fontsize=9)
fig.tight_layout(); plt.show()

外れ値なしでは第 1 主成分は $x_1$ 軸からほとんどずれないが、$(0,30)$ を 1 点加えるだけで
ほぼ $x_2$ 軸方向に変わる。$0.5\%$ の汚染で主成分が $85^\circ$ 以上回転している。
マハラノビス距離で下位 80% を残す簡易ロバスト推定なら、外れ値が 10 点（全体の 5%）でも
数度以内に収まる。ただしこの簡易法の破綻点は 0.2 なので、汚染が 20% を超えれば同じように壊れる。

§4.7 の対処は三つある。(a) ロバストな共分散推定（MCD は破綻点約 50%）、
(b) $L_1$ ノルム PCA、(c) 主成分追跡（$\boldsymbol{X}=\boldsymbol{L}_0+\boldsymbol{S}_0$ を核ノルム＋$L_1$ の凸緩和で分離、
定義4.18・式(4.11)・命題4.19、応用は注意4.20）。ここで実装したのは (a) の最も軽量な版である。

<a name="s6"></a>
## 4.6 スパース PCA：解釈性と説明力のトレードオフ

通常の PCA の負荷量は全成分が非零なので、$d$ が大きいと「この主成分は何を表すのか」を
言葉で説明できない。式(4.14)（Zou--Hastie--Tibshirani の定式化）は
命題4.22（PCA の自己回帰表現、式(4.13)）に elastic net 罰則を足したもので、
`sklearn.decomposition.SparsePCA` がこれを提供する。

20 変数だけが二つの潜在因子で動き、残り 40 変数はノイズという人工データ（$d=60$, $n=300$）で、
罰則の強さ $\alpha$ を振って**非零の個数**と**調整済み寄与率**の関係を見る（図4.4 に対応）。
調整済み寄与率は、スパース PCA では負荷量が直交せずスコアも無相関でなくなるため、
スコア行列を QR 分解して重複分を除いて計算する。

In [ ]:
from sklearn.decomposition import SparsePCA

rng = np.random.default_rng(0)
d, n, k = 60, 300, 2
Zl = rng.standard_normal((k, n))
Btrue = np.zeros((d, k))
Btrue[:10, 0] = 1.0            # 変数 0-9 が第1因子
Btrue[10:20, 1] = 1.0          # 変数 10-19 が第2因子
X = Btrue @ Zl + 0.4 * rng.standard_normal((d, n))       # d x n
Xt = X - X.mean(axis=1, keepdims=True)
tot_var = np.trace(Xt @ Xt.T) / n

def adjusted_ratio(B, Xt):
    """Zou らの調整済み寄与率：スコア行列を QR して重複を除く"""
    Zs = Xt.T @ B                                        # n x k（行がサンプル）
    R = np.linalg.qr(Zs, mode="r")
    return float((np.diag(R) ** 2).sum() / Xt.shape[1] / tot_var)

U = np.linalg.svd(Xt, full_matrices=False)[0][:, :k]
print(f"全分散 tr(Sigmahat) = {tot_var:.4f}")
print(f"通常の PCA        : 非零 {int((np.abs(U) > 1e-8).sum()):>3}/{d*k},"
      f"  寄与率 = {adjusted_ratio(U, Xt):.4f},"
      f"  真の台との一致 = {np.abs(U[:20]).sum()/np.abs(U).sum():.4f}")

rows = []
for alpha in (0.5, 1.0, 2.0, 4.0, 8.0, 16.0):
    sp = SparsePCA(n_components=k, alpha=alpha, ridge_alpha=0.01,
                   max_iter=200, random_state=0).fit(Xt.T)   # sklearn は n x d 規約
    B = sp.components_.T                                     # d x k
    nz = np.linalg.norm(B, axis=1) > 1e-8
    Bn = B / np.maximum(np.linalg.norm(B, axis=0), 1e-12)
    ar = adjusted_ratio(Bn, Xt)
    frac = np.abs(Bn[:20]).sum() / max(np.abs(Bn).sum(), 1e-12)
    rows.append((alpha, int((np.abs(B) > 1e-8).sum()), ar, frac, Bn))
    print(f"スパース PCA a={alpha:>4.1f}: 非零 {rows[-1][1]:>3}/{d*k},"
          f"  調整済み寄与率 = {ar:.4f},  真の台との一致 = {frac:.4f}")

fig, axes = plt.subplots(1, 3, figsize=(13, 3.9))
ax = axes[0]
for j, col in zip(range(k), (C["blue"], C["red"])):
    ax.stem(np.arange(d), U[:, j], linefmt=col, markerfmt=" ", basefmt=" ",
            label=L(f"第{j+1}主成分", f"PC{j+1}"))
ax.axvspan(-0.5, 19.5, color=C["green"], alpha=0.10)
ax.set_xlabel(L("変数の番号", "variable index")); ax.set_ylabel(L("負荷量", "loading"))
ax.set_title(L("通常の PCA（全変数が非零）", "plain PCA (all nonzero)")); ax.legend(fontsize=8)

Bn = rows[1][4]                                          # alpha = 1.0
ax = axes[1]
for j, col in zip(range(k), (C["blue"], C["red"])):
    ax.stem(np.arange(d), Bn[:, j], linefmt=col, markerfmt=" ", basefmt=" ",
            label=L(f"第{j+1}成分", f"comp {j+1}"))
ax.axvspan(-0.5, 19.5, color=C["green"], alpha=0.10)
ax.set_xlabel(L("変数の番号", "variable index")); ax.set_ylabel(L("負荷量", "loading"))
ax.set_title(L(r"スパース PCA（$\alpha=1$）", r"sparse PCA ($\alpha=1$)")); ax.legend(fontsize=8)

ax = axes[2]
ax.plot([r[1] for r in rows], [r[2] for r in rows], "o-", color=C["purple"], lw=2)
for r in [rows[0], rows[1], rows[-1]]:
    ax.annotate(rf"$\alpha$={r[0]}", (r[1], r[2]), textcoords="offset points",
                xytext=(-6, -14), fontsize=9)
ax.axhline(adjusted_ratio(U, Xt), color=C["gray"], ls="--", lw=1.6,
           label=L("通常の PCA", "plain PCA"))
ax.set_xlabel(L("非零な負荷量の個数", "number of nonzero loadings"))
ax.set_ylabel(L("調整済み寄与率", "adjusted variance ratio"))
ax.set_title(L("解釈性と説明力のトレードオフ", "interpretability vs explained variance"))
ax.legend(fontsize=9)
fig.tight_layout(); plt.show()

通常の PCA では 120 個すべての負荷量が非零で、真に効いている 20 変数の外側（変数 20〜59）にも
小さな係数が散らばる。スパース PCA は $\alpha$ を上げると非零の個数が減り、
負荷量が真の台（変数 0〜19、図の緑の帯）にほぼ限定される。

代償は説明力である。制約が増えるので $\boldsymbol{w}^\top\widehat{\boldsymbol{\Sigma}}\boldsymbol{w}$ は必ず通常の PCA 以下になる。
ただしこの人工データでは、罰則が「真に 0 の係数」を削っているあいだの代償はごく小さい——
非零が $120\to20$ に減っても調整済み寄与率は $0.6869\to0.6843$ しか下がらない。
代償がはっきり現れるのは罰則が**真の台まで削り始めてから**で、$\alpha=16$ では非零 17 個、
調整済み寄与率 $0.5313$ と急落する（右の図の左下の点）。
実データでは変数が互いに相関しているので、この落ち込みはもっと早く始まる。
§4.8 の警告のとおり、スパース PCA では負荷量の**直交性**とスコアの**無相関性**も失われるので、
寄与率は単純な固有値の比では定義できず、上のように調整が必要である。
また $L_1$ 罰則つき分散最大化の式(4.12) も式(4.14) もいずれも非凸で、得られるのは局所解である。

<a name="ex"></a>
## 演習

`# TODO` を埋めて実行せよ。解答は次節にある。

**演習 1（定理4.6・演習4.2：べき乗法の収束）**
$\boldsymbol{A}=\begin{pmatrix}3&1\\1&3\end{pmatrix}$（固有値 4 と 2）に $\boldsymbol{w}^{(0)}=(1,0)^\top$ から
べき乗法を適用し、Rayleigh 商の誤差 $4-\rho_t$ が $|\lambda_2/\lambda_1|^{2t}=0.25^t$ の速さで
減ることを確かめよ。さらに $\lambda_1=10$, $\lambda_2=9$, $\tan\theta_0=1$ のとき
$\sin\theta_t\le10^{-6}$ を保証する反復回数を式(4.3)から求め、
シフト $c=4.5$（$\lambda_d=0$ が既知）を使った場合と比べよ。

**演習 2（定理4.25・式(4.15)・演習4.4：Davis--Kahan）**
$\boldsymbol{\Sigma}=\operatorname{diag}(5,4.9,1)$、$\|\boldsymbol{E}\|_F=0.05$ の摂動を 200 通り作り、
$k=1$ と $k=2$ の部分空間について $\|\sin\Theta\|_F$ を実測して界
$2\|\boldsymbol{E}\|_F/\delta$ と比べよ。どちらを報告すべきか論じよ。

**演習 3（例4.23、§4.9：条件数の 2 乗）**
特異値が $\{1,s\}$ の $2\times50$ 行列で、小さいほうの特異値を
(i) 直接 SVD、(ii) $\widetilde{\boldsymbol{X}}\widetilde{\boldsymbol{X}}^\top$ の固有値の平方根
として求めたときの相対誤差を $s=10^{-4},\dots,10^{-10}$ で比べよ。

In [ ]:
# ---- 演習 1 ----
A = np.array([[3.0, 1.0], [1.0, 3.0]])
w = np.array([1.0, 0.0])
print("[演習1] べき乗法")
for t in range(1, 6):
    w = A @ w
    w = w / np.linalg.norm(w)
    rho = None      # TODO: Rayleigh 商 w^T A w
    print(f"  t={t}: w = {np.round(w, 4)},  4 - rho_t = {rho}")
# TODO: 0.9^t <= 1e-6 から反復回数を、シフト c=4.5 のときは (4.5/5.5)^t <= 1e-6 から求める
t_plain, t_shift = None, None
print("  必要な反復回数: シフトなし", t_plain, " シフト c=4.5", t_shift)

# ---- 演習 2 ----
Sig = np.diag([5.0, 4.9, 1.0])
rng = np.random.default_rng(0)
# TODO: ||E||_F = 0.05 の対称摂動を 200 通り作り、k=1,2 の sinTheta を測って界と比べる

# ---- 演習 3 ----
rng = np.random.default_rng(0)
Q2, _ = np.linalg.qr(rng.standard_normal((2, 2)))
V50, _ = np.linalg.qr(rng.standard_normal((50, 2)))
for s in (1e-4, 1e-6, 1e-8, 1e-9, 1e-10):
    X = (Q2 * np.array([1.0, s])) @ V50.T          # 特異値 {1, s} の 2 x 50 行列
    # TODO: (i) SVD 経由と (ii) X X^T の固有値経由で小さい特異値を求め、相対誤差を比べる
    pass

<a name="sol"></a>
## 演習の解答

In [ ]:
# ---- 演習 1 の解答 ----
A = np.array([[3.0, 1.0], [1.0, 3.0]])
w = np.array([1.0, 0.0])
print("[演習1] べき乗法（lambda_1=4, lambda_2=2）")
prev = None
for t in range(1, 6):
    w = A @ w; w = w / np.linalg.norm(w)
    rho = float(w @ A @ w)
    gap = 4 - rho
    print(f"  t={t}: w = {np.round(w, 4)},  4 - rho_t = {gap:.6f}"
          + (f",  前ステップとの比 = {gap/prev:.4f}" if prev else ""))
    prev = gap
print("  |lambda_2/lambda_1|^2 = 0.25 なので比はおよそ 0.25 になる")

t_plain = int(np.ceil(6 * np.log(10) / np.log(1 / 0.9)))
t_shift = int(np.ceil(6 * np.log(10) / np.log(5.5 / 4.5)))
print(f"  lambda_1=10, lambda_2=9: sin theta_t <= 1e-6 に必要な反復 = {t_plain} 回")
print(f"  シフト c=4.5（比 4.5/5.5 = {4.5/5.5:.4f}）なら {t_shift} 回（約半分）")

# ---- 演習 2 の解答 ----
Sig = np.diag([5.0, 4.9, 1.0])
rng = np.random.default_rng(0)
nrmE = 0.05
res = {1: [], 2: []}
for _ in range(200):
    E = rng.standard_normal((3, 3)); E = (E + E.T) / 2
    E *= nrmE / np.linalg.norm(E, "fro")
    Vh = np.linalg.eigh(Sig + E)[1][:, ::-1]
    for kk in (1, 2):
        P = np.eye(3)[:, :kk] @ np.eye(3)[:, :kk].T          # 真の部分空間の射影
        Ph = Vh[:, :kk] @ Vh[:, :kk].T
        res[kk].append(np.linalg.norm(P - Ph, "fro") / np.sqrt(2))
print("\n[演習2] Davis--Kahan（||E||_F = 0.05）")
for kk, delta in ((1, 5.0 - 4.9), (2, 4.9 - 1.0)):
    bound = 2 * nrmE / delta
    print(f"  k={kk}: ギャップ delta = {delta:.2f},  界 2||E||_F/delta = {bound:.4f},"
          f"  実測 ||sin Theta||_F の最大 = {max(res[kk]):.4f}, 中央値 = {np.median(res[kk]):.4f}")
print("  k=1 の界は 1.0 で sin Theta <= 1 は自明、すなわち何も言えない（実際に大きく回る）。")
print("  k=2 の界は 0.0256 で部分空間の安定性を正しく予言する。")
print("  → 報告すべきは「第1主成分の方向」ではなく「上位2主成分が張る部分空間」である。")

fig, ax = plt.subplots(figsize=(6.2, 3.8))
ax.hist(res[1], bins=25, color=C["red"], alpha=0.7, label=L("k=1 の実測", "k=1 measured"))
ax.hist(res[2], bins=25, color=C["blue"], alpha=0.7, label=L("k=2 の実測", "k=2 measured"))
ax.axvline(2 * nrmE / 0.1, color=C["red"], ls="--", lw=2, label=L("k=1 の界 (1.0)", "k=1 bound"))
ax.axvline(2 * nrmE / 3.9, color=C["blue"], ls="--", lw=2, label=L("k=2 の界 (0.026)", "k=2 bound"))
ax.set_xscale("log"); ax.set_xlabel(L(r"$\|\sin\Theta\|_F$", r"$\|\sin\Theta\|_F$"))
ax.set_ylabel(L("回数", "count")); ax.legend(fontsize=8)
ax.set_title(L("ギャップが分母に来る", "the gap sits in the denominator"))
fig.tight_layout(); plt.show()

# ---- 演習 3 の解答 ----
rng = np.random.default_rng(0)
Q2, _ = np.linalg.qr(rng.standard_normal((2, 2)))
V50, _ = np.linalg.qr(rng.standard_normal((50, 2)))
print("\n[演習3] 条件数の 2 乗（小さい特異値の相対誤差）")
print("      s        SVD 経由      Sigmahat 経由")
for s in (1e-4, 1e-6, 1e-8, 1e-9, 1e-10):
    X = (Q2 * np.array([1.0, s])) @ V50.T
    s_svd = np.linalg.svd(X, compute_uv=False)[1]
    ev = np.linalg.eigvalsh(X @ X.T)
    s_cov = np.sqrt(max(ev.min(), 0.0))
    print(f"  {s:8.0e}   {abs(s_svd - s)/s:11.2e}   {abs(s_cov - s)/s:11.2e}")
print("  kappa(Xtilde) = 1/s なので Sigmahat の条件数は 1/s^2。")
print("  倍精度の eps ~ 2.2e-16 だから 1/s^2 > 1e16、すなわち s < 1e-8 で共分散経由は破綻する。")